In [ ]:
import numpy as np
import os, sys, re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from MAE_model_downstream import PedSleepMAE
from utils.misc import setup_seed
from dataloader import HDF5Dataset

search_label = "sleep_label"
directory_path = os.path.join(os.path.dirname(os.getcwd()), "PYTORCH", "hdf5")
patch_size, mask_ratio, emb_dim, num_head, num_layer = 8, 15, 64, 4, 3
needed_patient_IDs, needed_study_IDs = ["xxxx"], ["xxxx"]
seed = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
num_patches = int(3840 / patch_size)
setup_seed(seed)

def extract_sample_id(f):
    m = re.search(r"_sample_(\d+)\.hdf5$", f)
    return int(m.group(1)) if m else float("inf")

files = [os.path.join(directory_path, x) for x in os.listdir(directory_path) if x.endswith(".hdf5")]
files = [f for f in files if f.split("/")[-1].split("_")[0] in needed_patient_IDs and f.split("/")[-1].split("_")[1] in needed_study_IDs]
sorted_files = sorted(files, key=extract_sample_id)

if len(sorted_files) == 0:
    print("Invalid ID"); sys.exit(1)

print(f"Total sorted files: {len(sorted_files)}")

model = PedSleepMAE(
    batch_size=len(sorted_files),
    patch_size=patch_size,
    mask_ratio=mask_ratio,
    emb_dim=emb_dim,
    num_head=num_head,
    num_layer=num_layer,
).to(device)

ckpt_file = f"../savedmodels{mask_ratio}/signalmask{mask_ratio}_patch_size{patch_size}.pt"
ckpt = torch.load(ckpt_file, weights_only=True)
model.load_state_dict(ckpt["state_dict"])

pool = nn.AdaptiveMaxPool1d(1)

dataset = HDF5Dataset(sorted_files, search_label)
loader = DataLoader(dataset, batch_size=100, shuffle=False)

save_dir = "output_embeddings_sorted"
os.makedirs(save_dir, exist_ok=True)

embeddings, labels = [], []

for i, (signal, label, _) in enumerate(loader):
    print(f"Batch {i}")
    with torch.no_grad():
        signal = signal.squeeze().float().to(device)
        label = label.squeeze().cpu().numpy()
        n, _, _ = signal.shape

        enc, _ = model.encoder(signal)
        enc = enc[:, :, 1:, :].reshape(-1, num_patches, emb_dim)
        pooled = pool(enc).reshape(n, -1)

        embeddings.extend(pooled.cpu().numpy())
        labels.extend(label)

emb = np.vstack(embeddings)
lab = np.array(labels)

prefix = f"{'_'.join(needed_patient_IDs)}_{'_'.join(needed_study_IDs)}"
np.save(os.path.join(save_dir, f"{prefix}_embeddings.npy"), emb)
np.save(os.path.join(save_dir, f"{prefix}_labels.npy"), lab)

print(f"Saved {prefix}_embeddings.npy {emb.shape}")
print(f"Saved {prefix}_labels.npy {lab.shape}")

In [ ]:
import numpy as np
import phate
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

embeddings = np.load("embeddings.npy")
sleep_stages = np.load("labels.npy")
time_indices = np.arange(len(embeddings))

sleep_stage_names = {
    0: "Wake",
    1: "N1",
    2: "N2",
    3: "N3",
    4: "REM"
}

time_cmap = plt.get_cmap("turbo")
stage_palette = sns.color_palette("Set1", 5)
stage_colors = [stage_palette[int(stage)] for stage in sleep_stages]

pca = PCA(n_components=2)
pca_features = pca.fit_transform(embeddings)

tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
tsne_features = tsne.fit_transform(embeddings)

phate_operator = phate.PHATE(n_components=2, knn=5, t=12, n_pca=100, random_state=42)
phate_features = phate_operator.fit_transform(embeddings)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

scatter1 = axes[0, 0].scatter(pca_features[:, 0], pca_features[:, 1], c=time_indices, cmap=time_cmap, alpha=0.8, s=10)
axes[0, 0].set_title("PCA (Progression Coloring)")
fig.colorbar(scatter1, ax=axes[0, 0], label="Time Index")

scatter2 = axes[0, 1].scatter(tsne_features[:, 0], tsne_features[:, 1], c=time_indices, cmap=time_cmap, alpha=0.8, s=10)
axes[0, 1].set_title("t-SNE (Progression Coloring)")
fig.colorbar(scatter2, ax=axes[0, 1], label="Time Index")

scatter3 = axes[0, 2].scatter(phate_features[:, 0], phate_features[:, 1], c=time_indices, cmap=time_cmap, alpha=0.8, s=10)
axes[0, 2].set_title("PHATE (Progression Coloring)")
fig.colorbar(scatter3, ax=axes[0, 2], label="Time Index")

scatter4 = axes[1, 0].scatter(pca_features[:, 0], pca_features[:, 1], c=stage_colors, alpha=0.8, s=10)
axes[1, 0].set_title("PCA (Categorical Labels)")

scatter5 = axes[1, 1].scatter(tsne_features[:, 0], tsne_features[:, 1], c=stage_colors, alpha=0.8, s=10)
axes[1, 1].set_title("t-SNE (Categorical Labels)")

scatter6 = axes[1, 2].scatter(phate_features[:, 0], phate_features[:, 1], c=stage_colors, alpha=0.8, s=10)
axes[1, 2].set_title("PHATE (Categorical Labels)")

legend_labels = [
    plt.Line2D([0], [0], marker='o', color='w', label=sleep_stage_names[i],
               markersize=10, markerfacecolor=stage_palette[i]) 
    for i in range(5)
]
fig.legend(handles=legend_labels, loc='upper right', title="Labels", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import phate
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

embeddings = np.load("embeddings.npy")
sleep_stages = np.load("labels.npy")
time_indices = np.arange(len(embeddings))

sleep_stage_names = {
    0: "Wake",
    4: "REM",
    1: "N1",
    2: "N2",
    3: "N3"
}
ordered_stages = [0, 4, 1, 2, 3]

time_cmap = sns.color_palette("turbo", as_cmap=True)
stage_palette = sns.color_palette("coolwarm", len(ordered_stages))
stage_color_map = {stage: stage_palette[i] for i, stage in enumerate(ordered_stages)}
stage_colors = [stage_color_map[int(stage)] for stage in sleep_stages]

pca = PCA(n_components=2).fit_transform(embeddings)
tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42).fit_transform(embeddings)
phate_op = phate.PHATE(n_components=2, knn=5, t=8, n_pca=100, random_state=42)
phate_features = phate_op.fit_transform(embeddings)
umap_features = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(embeddings)

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

sc0 = axes[0, 0].scatter(pca[:, 0], pca[:, 1], c=time_indices, cmap=time_cmap, s=10, alpha=0.8)
axes[0, 0].set_title("PCA (Progression Coloring)")
axes[0, 1].scatter(tsne[:, 0], tsne[:, 1], c=time_indices, cmap=time_cmap, s=10, alpha=0.8)
axes[0, 1].set_title("t-SNE (Progression Coloring)")
axes[0, 2].scatter(phate_features[:, 0], phate_features[:, 1], c=time_indices, cmap=time_cmap, s=10, alpha=0.8)
axes[0, 2].set_title("PHATE (Progression Coloring)")
axes[0, 3].scatter(umap_features[:, 0], umap_features[:, 1], c=time_indices, cmap=time_cmap, s=10, alpha=0.8)
axes[0, 3].set_title("UMAP (Progression Coloring)")

axes[1, 0].scatter(pca[:, 0], pca[:, 1], c=stage_colors, s=10, alpha=0.8)
axes[1, 0].set_title("PCA (Categorical Labels)")
axes[1, 1].scatter(tsne[:, 0], tsne[:, 1], c=stage_colors, s=10, alpha=0.8)
axes[1, 1].set_title("t-SNE (Categorical Labels)")
axes[1, 2].scatter(phate_features[:, 0], phate_features[:, 1], c=stage_colors, s=10, alpha=0.8)
axes[1, 2].set_title("PHATE (Categorical Labels)")
axes[1, 3].scatter(umap_features[:, 0], umap_features[:, 1], c=stage_colors, s=10, alpha=0.8)
axes[1, 3].set_title("UMAP (Categorical Labels)")

cbar_ax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
cb = fig.colorbar(sc0, cax=cbar_ax)
cb.set_label('Time Index', fontsize=12)

legend_labels = [
    plt.Line2D([0], [0], marker='o', color='w', label=sleep_stage_names[i],
               markersize=10, markerfacecolor=stage_palette[idx]) 
    for idx, i in enumerate(ordered_stages)
]
fig.legend(handles=legend_labels, loc='lower right', title="Labels", fontsize=12)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sleep_stages = np.load("labels.npy")
time_indices = np.arange(len(sleep_stages)) * 0.5  # each index = 30 seconds → minutes

stage_labels = ['Wake', 'N1', 'N2', 'N3', 'REM']
stage_ticks = [0, 1, 2, 3, 4]
stage_display_order = [0, 4, 1, 2, 3]

display_mapping = {0: 0, 1: 2, 2: 3, 3: 4, 4: 1}
stage_display_values = np.array([display_mapping[int(s)] for s in sleep_stages])

plt.figure(figsize=(15, 4))
plt.step(time_indices, stage_display_values, where='mid', linewidth=1.8, color='darkslateblue')
plt.yticks(
    [display_mapping[i] for i in stage_display_order],
    [stage_labels[i] for i in stage_display_order]
)
plt.xlabel("Time (minutes)", fontsize=12)
plt.ylabel("Stage", fontsize=12)
plt.title("Stages Over Time", fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.ylim(-0.5, 4.5)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sleep_stages = np.load("labels.npy")
time_indices = np.arange(len(sleep_stages)) * 0.5  # 30s intervals → minutes

stage_names_ordered = ['Wake', 'REM', 'N1', 'N2', 'N3']
stage_value_map = {0: 0, 4: 1, 1: 2, 2: 3, 3: 4}
stage_colors = {
    0: '#ff6f69',
    1: '#ffeead',
    2: '#96ceb4',
    3: '#379683',
    4: '#88d8b0',
}

mapped_y = [stage_value_map[int(s)] for s in sleep_stages]

plt.figure(figsize=(15, 3))
for i in range(len(sleep_stages)):
    orig_stage = int(sleep_stages[i])
    y_val = stage_value_map[orig_stage]
    plt.hlines(y=y_val, xmin=time_indices[i], xmax=time_indices[i] + 0.5,
               colors=stage_colors[orig_stage], linewidth=8)

plt.yticks(ticks=[0, 1, 2, 3, 4], labels=stage_names_ordered)
plt.ylim(-0.5, 4.5)
plt.xlabel("Time (minutes)", fontsize=12)
plt.title("Stages Over Time", fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, homogeneity_score
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

embeddings = np.load("embeddings.npy")
sleep_stages = np.load("labels.npy")
time_indices = np.arange(len(embeddings))

scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
tsne_features = tsne.fit_transform(embeddings_scaled)

silhouette = silhouette_score(tsne_features, sleep_stages)
davies_bouldin = davies_bouldin_score(tsne_features, sleep_stages)
calinski_harabasz = calinski_harabasz_score(tsne_features, sleep_stages)

print("t-SNE Metrics:")
print(f"   - Silhouette Score: {silhouette:.3f} (Higher is better)")
print(f"   - Davies-Bouldin Index: {davies_bouldin:.3f} (Lower is better)")
print(f"   - Calinski-Harabasz Index: {calinski_harabasz:.3f} (Higher is better)\n")

sleep_time_corr, _ = spearmanr(sleep_stages, time_indices)
print(f"Spearman Correlation (Stages vs. Time): {sleep_time_corr:.3f}")

mi_tsne = mutual_info_classif(tsne_features, sleep_stages, discrete_features=False).mean()
print(f"Mutual Information (Higher is better): {mi_tsne:.3f}\n")

def temporal_coherence_score(embedding_features, time_labels, k_neighbors=10):
    neigh = NearestNeighbors(n_neighbors=k_neighbors)
    neigh.fit(embedding_features)
    _, indices = neigh.kneighbors(embedding_features)
    avg_time_variance = np.mean([np.var(time_labels[neighbors]) for neighbors in indices])
    return avg_time_variance

tsne_temporal_score = temporal_coherence_score(tsne_features, time_indices)
print(f"Temporal Coherence Score (Lower is better): {tsne_temporal_score:.3f}")

tsne_purity = homogeneity_score(sleep_stages, KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(tsne_features))
print(f"Cluster Purity Score (Higher is better): {tsne_purity:.3f}")